# Level 2: agent A vs agent B

Agent A uses raw tables, join specs, and SQL snippets. Agent B uses metric views plus two dimensions. Entity matching is added on plant, line, and defect code. Entity matching cannot be used on columns that have row filters or masks.


## Bind config


In [ ]:
import sys
from pathlib import Path

dbutils.widgets.text("catalog", "YOUR_CATALOG", "Catalog")
dbutils.widgets.text("schema", "YOUR_SCHEMA", "Schema")
dbutils.widgets.text("warehouse_id", "YOUR_WAREHOUSE_ID", "SQL warehouse id")
dbutils.widgets.text("telemetry_rows", "10000000", "Telemetry rows (10000000 or 50000000)")
dbutils.widgets.text("repo_root", "", "Workspace folder that contains shared/")

here = Path.cwd().resolve()
root = None
repo_root_w = dbutils.widgets.get("repo_root").strip()
if repo_root_w and (Path(repo_root_w) / "shared" / "config.py").exists():
    root = Path(repo_root_w)
if root is None:
    for candidate in [here, *here.parents]:
        if (candidate / "shared" / "config.py").exists():
            root = candidate
            break
if root is None:
    raise FileNotFoundError("shared/config.py not found. Set repo_root to the workspace folder that contains shared/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.config import load_config
from shared.lib.genie_client import GenieClient

cfg = load_config()
print("catalog", cfg.catalog)
print("schema", cfg.schema)
print("fqn", cfg.fqn)


## Create agent A (tables) and agent B (metric views)


In [ ]:
client = GenieClient(warehouse_id=cfg.warehouse_id)
tables_a = [
    cfg.table("sensor_telemetry"),
    cfg.table("fct_line_shift"),
    cfg.table("maintenance_work_orders"),
    cfg.table("equipment"),
    cfg.table("bom"),
    cfg.table("bom_component_bridge"),
]
tables_b = [
    cfg.table("mv_line_performance"),
    cfg.table("mv_quality"),
    cfg.table("mv_maintenance"),
    cfg.table("fct_line_shift"),
    cfg.table("maintenance_work_orders"),
]
existing = {s.get("title"): s for s in client.list_spaces()}

def ensure(title, desc, tables):
    if title in existing:
        return existing[title]
    return client.create_space(title, desc, tables)

a = ensure("Apex L2 A (tables)", "Raw tables plus join specs", tables_a)
b = ensure("Apex L2 B (metric views)", "Metric views for OEE FPY MTBF", tables_b)
a_id = a.get("space_id") or a.get("id")
b_id = b.get("space_id") or b.get("id")

def patch_b(space: dict) -> dict:
    space.setdefault("instructions", {})["text_instructions"] = [
        {
            "id": "l2b-text",
            "content": [
                "Prefer MEASURE() on metric views. OEE is a 0 to 1 ratio of sums. Do not average daily OEE. FPY is good_units / units."
            ],
        }
    ]
    return space

client.merge_serialized_space(b_id, patch_b)
spark.sql(
    f"""CREATE OR REPLACE TABLE {cfg.table('agent_ids_l2')} AS
    SELECT '{a_id}' AS agent_a_id, '{b_id}' AS agent_b_id"""
)
print("A", client.ui_space_url(a_id))
print("B", client.ui_space_url(b_id))
